# 从零实现 HGT 风格异构图网络：关系级多头消息与类型级输出

本 Notebook 只使用 PyTorch 基础张量与 `nn.Module`，显式实现 relation-specific Q/K/V/message、按目标节点归一化的稳定 softmax、多头聚合、type-specific output projection 与异构节点分类；不使用 PyG、DGL、`nn.MultiheadAttention` 或现成 GNN/Transformer 层。

学习目标是一条可审计的工程链路：冻结异构 schema → 租户隔离 → 有向关系消息 → 空邻居/残差语义 → 数值 oracle → 按 tenant 切分的受控训练 → 带外发布信任锚。数据是离线合成 fixture，只用于证明实现和协议正确，不冒充真实业务泛化结果。

原始思想参考：[Heterogeneous Graph Transformer, WWW 2020](https://arxiv.org/abs/2003.01332)。相关的关系参数化与异构注意力背景可对照 [R-GCN](https://arxiv.org/abs/1703.06103) 和 [Heterogeneous Graph Attention Network](https://arxiv.org/abs/1903.07293)。教学实现保留 HGT 的类型/关系参数化核心，并刻意缩小数据与算子规模。


In [ ]:
from __future__ import annotations

import copy
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)
from dataclasses import dataclass
import hashlib
import json
import math
import random
from types import MappingProxyType

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 4901
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
DTYPE = torch.float32

def canonical_digest(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def tensor_state_digest(state: dict[str, torch.Tensor]) -> str:
    h = hashlib.sha256()
    for key in sorted(state):
        value = state[key].detach().cpu().contiguous()
        h.update(key.encode("utf-8"))
        h.update(str(value.dtype).encode("ascii"))
        h.update(str(tuple(value.shape)).encode("ascii"))
        h.update(value.numpy().tobytes())
    return h.hexdigest()

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1
assert torch.initial_seed() == SEED
assert tensor_state_digest({"x": torch.tensor([1.0])}) != tensor_state_digest({"x": torch.tensor([2.0])})
assert not any(name in globals() for name in ("torch_geometric", "dgl"))


## 1. Schema 不是注释，而是模型输入合同

关系用三元组 `(source_type, relation_name, target_type)` 标识。本例冻结三种有向关系：`user-click-item`、`item-rev_click-user`、`user-follow-user`。同名节点编号只在各自类型内部有效，不能把 user 的局部编号拿去索引 item。

每个节点还带 `tenant_id`。任何边都必须满足源、目标属于同一 tenant；否则即使索引合法，也会造成跨客户信息泄漏。显式 `follow` 自环被拒绝，因为中心节点信息已经由 residual 路径处理，混入自环会重复计权。


In [ ]:
SchemaEdge = tuple[str, str, str]
SCHEMA: tuple[SchemaEdge, ...] = (
    ("user", "click", "item"),
    ("item", "rev_click", "user"),
    ("user", "follow", "user"),
)
NODE_TYPES = tuple(sorted({t for edge in SCHEMA for t in (edge[0], edge[2])}))

def relation_key(edge: SchemaEdge) -> str:
    return "__".join(edge)

def validate_schema(schema: tuple[SchemaEdge, ...]) -> None:
    if not schema or len(schema) != len(set(schema)):
        raise ValueError("schema 不能为空且关系三元组不能重复")
    for src, rel, dst in schema:
        if not all(isinstance(v, str) and v and "__" not in v for v in (src, rel, dst)):
            raise ValueError("类型/关系名称非法")

validate_schema(SCHEMA)
assert NODE_TYPES == ("item", "user")
assert len({relation_key(e) for e in SCHEMA}) == len(SCHEMA)

try:
    validate_schema((SCHEMA[0], SCHEMA[0]))
    raise AssertionError("重复 schema 未被拒绝")
except ValueError as exc:
    assert "重复" in str(exc)


## 2. softmax 必须按“目标节点 × head”分组

对边 $e=(s\rightarrow t)$ 和 head $h$，注意力分数为

$$a_{e,h}=\frac{Q_r(x_t)\cdot K_r(x_s)}{\sqrt{d_h}}\,\mu_{r,h}.$$

归一化集合只能是同一目标节点的所有入边（可以来自不同 relation）：

$$\alpha_{e,h}=\frac{\exp(a_{e,h}-m_{t,h})}{\sum_{e':dst(e')=t}\exp(a_{e',h}-m_{t,h})}.$$

减去组内最大值避免 `exp(1000)` 溢出。按 relation 分别 softmax 会错误地让每种关系都获得总质量 1。


In [ ]:
def stable_target_softmax(logits: torch.Tensor, dst: torch.Tensor, num_targets: int) -> torch.Tensor:
    if logits.ndim != 2 or dst.ndim != 1 or logits.shape[0] != dst.numel():
        raise ValueError("logits/dst shape 合同不匹配")
    if dst.dtype != torch.long or num_targets < 0:
        raise ValueError("dst 必须是 long，num_targets 必须非负")
    if not torch.isfinite(logits).all():
        raise ValueError("attention logits 含非有限值")
    if dst.numel() and (int(dst.min()) < 0 or int(dst.max()) >= num_targets):
        raise ValueError("目标节点索引越界")
    out = torch.empty_like(logits)
    for target in dst.unique(sorted=True).tolist():
        mask = dst == target
        group = logits[mask]
        shifted = group - group.max(dim=0, keepdim=True).values
        out[mask] = shifted.exp() / shifted.exp().sum(dim=0, keepdim=True)
    return out

probe_logits = torch.tensor([[1000.0, -1000.0], [999.0, -999.0], [-500.0, 500.0]])
probe_dst = torch.tensor([0, 0, 1], dtype=torch.long)
probe_alpha = stable_target_softmax(probe_logits, probe_dst, 2)
assert torch.isfinite(probe_alpha).all()
assert torch.allclose(probe_alpha[:2].sum(0), torch.ones(2), atol=1e-6)
assert torch.allclose(probe_alpha[2], torch.ones(2), atol=1e-6)
assert float(probe_alpha[0, 0]) > float(probe_alpha[1, 0])

try:
    stable_target_softmax(torch.tensor([[float("nan")]]), torch.tensor([0]), 1)
    raise AssertionError("NaN logits 未被拒绝")
except ValueError as exc:
    assert "非有限" in str(exc)


## 3. 手写 relation-specific 多头消息层

每个 relation 独立拥有 $Q_r,K_r,V_r,M_r$，因此反向边不是正向边的别名。消息为

$$m_{e,h}=M_r(V_r(x_s))_h,\qquad \tilde h_t=\Vert_h\sum_{e:dst(e)=t}\alpha_{e,h}m_{e,h}.$$

聚合后使用 target type 专属的 $W^O_{\tau(t)}$。残差门控为

$$h'_t=\sigma(g_{\tau})\,\mathrm{GELU}(W^O_{\tau}\tilde h_t)+(1-\sigma(g_{\tau}))h_t.$$

若节点没有入边，`aggregate=0` 且无 bias，因此输出严格等于 residual 分支，而不是 NaN 或任意 bias。


In [ ]:
class HGTLayer(nn.Module):
    def __init__(self, schema: tuple[SchemaEdge, ...], d_model: int, num_heads: int):
        super().__init__()
        validate_schema(schema)
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads:
            raise ValueError("d_model 必须能被正 head 数整除")
        self.schema = tuple(schema)
        self.node_types = tuple(sorted({t for e in schema for t in (e[0], e[2])}))
        self.d_model, self.num_heads, self.head_dim = d_model, num_heads, d_model // num_heads
        keys = [relation_key(e) for e in schema]
        self.q_rel = nn.ModuleDict({k: nn.Linear(d_model, d_model, bias=False) for k in keys})
        self.k_rel = nn.ModuleDict({k: nn.Linear(d_model, d_model, bias=False) for k in keys})
        self.v_rel = nn.ModuleDict({k: nn.Linear(d_model, d_model, bias=False) for k in keys})
        self.msg_rel = nn.ModuleDict({k: nn.Linear(d_model, d_model, bias=False) for k in keys})
        self.rel_prior = nn.ParameterDict({k: nn.Parameter(torch.ones(num_heads)) for k in keys})
        self.out_type = nn.ModuleDict({t: nn.Linear(d_model, d_model, bias=False) for t in self.node_types})
        self.skip_type = nn.ParameterDict({t: nn.Parameter(torch.zeros(1)) for t in self.node_types})

    def _validate(self, x_by_type, edges, tenant_by_type):
        if set(x_by_type) != set(self.node_types) or set(tenant_by_type) != set(self.node_types):
            raise ValueError("节点类型集合与冻结 schema 不一致")
        allowed = set(self.schema)
        if not set(edges).issubset(allowed):
            raise ValueError("输入包含冻结 schema 之外的关系/类型")
        base_device, base_dtype = None, None
        for node_type in self.node_types:
            x, tenant = x_by_type[node_type], tenant_by_type[node_type]
            if (x.ndim != 2 or x.shape[1] != self.d_model or x.shape[0] == 0
                    or not torch.is_floating_point(x) or not torch.isfinite(x).all()):
                raise ValueError("节点特征必须是非空有限 [N_type,D]")
            if tenant.ndim != 1 or tenant.dtype != torch.long or tenant.numel() != x.shape[0]:
                raise ValueError("tenant 张量合同不匹配")
            if bool((tenant < 0).any()): raise ValueError("tenant id 必须非负")
            base_device = x.device if base_device is None else base_device
            base_dtype = x.dtype if base_dtype is None else base_dtype
            if x.device != base_device or tenant.device != base_device or x.dtype != base_dtype:
                raise ValueError("所有特征、tenant 与边必须位于同一设备且特征 dtype 一致")
        for edge, edge_index in edges.items():
            src_type, _, dst_type = edge
            if edge_index.ndim != 2 or edge_index.shape[0] != 2 or edge_index.dtype != torch.long:
                raise ValueError("edge_index 必须是 long[2,E]")
            if edge_index.device != base_device:
                raise ValueError("edge_index 设备不一致")
            if edge_index.numel():
                src, dst = edge_index
                if int(src.min()) < 0 or int(src.max()) >= x_by_type[src_type].shape[0]:
                    raise ValueError("源节点索引越界")
                if int(dst.min()) < 0 or int(dst.max()) >= x_by_type[dst_type].shape[0]:
                    raise ValueError("目标节点索引越界")
                if src_type == dst_type and bool((src == dst).any()):
                    raise ValueError("显式同类型自环会与 residual 重复计权")
                if not torch.equal(tenant_by_type[src_type][src], tenant_by_type[dst_type][dst]):
                    raise ValueError("检测到跨 tenant 边")

    def forward(self, x_by_type, edges, tenant_by_type, return_attention: bool = False):
        self._validate(x_by_type, edges, tenant_by_type)
        aggregate = {t: torch.zeros_like(x_by_type[t]) for t in self.node_types}
        attention = {}
        for dst_type in self.node_types:
            score_parts, msg_parts, dst_parts, meta = [], [], [], []
            for edge in self.schema:
                src_type, _, target_type = edge
                if target_type != dst_type or edge not in edges or edges[edge].shape[1] == 0:
                    continue
                key = relation_key(edge)
                src, dst = edges[edge]
                e_count = src.numel()
                q = self.q_rel[key](x_by_type[dst_type][dst]).view(e_count, self.num_heads, self.head_dim)
                k = self.k_rel[key](x_by_type[src_type][src]).view(e_count, self.num_heads, self.head_dim)
                v = self.v_rel[key](x_by_type[src_type][src])
                msg = self.msg_rel[key](v).view(e_count, self.num_heads, self.head_dim)
                score = (q * k).sum(-1) * self.rel_prior[key] / math.sqrt(self.head_dim)
                score_parts.append(score); msg_parts.append(msg); dst_parts.append(dst)
                meta.append((edge, e_count))
            if score_parts:
                all_score, all_msg, all_dst = torch.cat(score_parts), torch.cat(msg_parts), torch.cat(dst_parts)
                alpha = stable_target_softmax(all_score, all_dst, x_by_type[dst_type].shape[0])
                weighted = (alpha.unsqueeze(-1) * all_msg).reshape(-1, self.d_model)
                aggregate[dst_type].index_add_(0, all_dst, weighted)
                offset = 0
                for edge, count in meta:
                    attention[edge] = alpha[offset:offset + count]
                    offset += count
        out = {}
        for node_type in self.node_types:
            gate = torch.sigmoid(self.skip_type[node_type])
            transformed = F.gelu(self.out_type[node_type](aggregate[node_type]))
            out[node_type] = gate * transformed + (1.0 - gate) * x_by_type[node_type]
        return (out, attention) if return_attention else out

assert issubclass(HGTLayer, nn.Module)
assert "forward" in HGTLayer.__dict__
assert HGTLayer(SCHEMA, 8, 2).head_dim == 4


## 4. 方向、关系参数、空邻居与隔离 oracle

下面不是只检查 shape：正向 click 只能改变 item 的消息分支，反向 rev_click 只能改变 user；跨 relation 参数对象必须独立；所有入边的注意力按目标节点求和为 1。空边图走精确 residual；显式自环、跨 tenant 边、非法 relation 均 fail-closed。


In [ ]:
torch.manual_seed(4902)
probe_layer = HGTLayer(SCHEMA, d_model=8, num_heads=2)
probe_x = {"user": torch.randn(4, 8), "item": torch.randn(2, 8)}
probe_tenant = {"user": torch.tensor([0, 0, 1, 1]), "item": torch.tensor([0, 1])}
click = ("user", "click", "item")
rev = ("item", "rev_click", "user")
follow = ("user", "follow", "user")
click_edges = {click: torch.tensor([[0, 1, 2, 3], [0, 0, 1, 1]], dtype=torch.long)}

click_out, click_attn = probe_layer(probe_x, click_edges, probe_tenant, return_attention=True)
empty_out = probe_layer(probe_x, {}, probe_tenant)
for node_type in NODE_TYPES:
    expected = (1.0 - torch.sigmoid(probe_layer.skip_type[node_type])) * probe_x[node_type]
    assert torch.allclose(empty_out[node_type], expected, atol=1e-7)
assert not torch.allclose(click_out["item"], empty_out["item"])
assert torch.allclose(click_out["user"], empty_out["user"], atol=1e-7)
assert click_attn[click].shape == (4, 2)
assert torch.allclose(click_attn[click][:2].sum(0), torch.ones(2), atol=1e-6)
assert torch.allclose(click_attn[click][2:].sum(0), torch.ones(2), atol=1e-6)
assert probe_layer.q_rel[relation_key(click)].weight.data_ptr() != probe_layer.q_rel[relation_key(rev)].weight.data_ptr()
assert probe_layer.out_type["user"].weight.data_ptr() != probe_layer.out_type["item"].weight.data_ptr()

rev_edges = {rev: torch.tensor([[0, 1], [0, 2]], dtype=torch.long)}
rev_out = probe_layer(probe_x, rev_edges, probe_tenant)
assert not torch.allclose(rev_out["user"], empty_out["user"])
assert torch.allclose(rev_out["item"], empty_out["item"], atol=1e-7)

bad_cases = [
    ({click: torch.tensor([[0], [1]])}, "跨 tenant"),
    ({follow: torch.tensor([[0], [0]])}, "自环"),
    ({("user", "unknown", "item"): torch.tensor([[0], [0]])}, "schema"),
]
for bad_edges, expected_text in bad_cases:
    try:
        probe_layer(probe_x, bad_edges, probe_tenant)
        raise AssertionError("非法异构边未被拒绝")
    except ValueError as exc:
        assert expected_text in str(exc)

try:
    probe_layer({"user": probe_x["user"].long(), "item": probe_x["item"]}, {}, probe_tenant)
    raise AssertionError("整数节点特征未被拒绝")
except ValueError as exc:
    assert "节点特征" in str(exc)


## 5. 按 tenant 切分的受控异构节点分类

每个 tenant 有 2 个 user 和 1 个 item。item 类别由点击它的 user 局部模式决定；item 自身只含常量特征，因此模型必须使用 `user→item` 关系。tenant 0–19/20–23/24–27 分别作为 train/validation/test，任何 tenant 不跨 split。

这仍是规则化 fixture：测试 tenant 与训练 tenant 来自同一生成机制。高准确率只能说明关系消息和切分链路可运行，不能声称对真实冷启动租户泛化。


In [ ]:
@dataclass(frozen=True)
class HeteroSnapshot:
    split: str
    tenant_ids: tuple[int, ...]
    x_by_type: dict[str, torch.Tensor]
    tenant_by_type: dict[str, torch.Tensor]
    edges: dict[SchemaEdge, torch.Tensor]
    item_labels: torch.Tensor
    snapshot_digest: str

def snapshot_digest(tensors: dict[str, torch.Tensor], metadata: dict) -> str:
    state_part = tensor_state_digest(tensors)
    return canonical_digest({"tensor_digest": state_part, "metadata": metadata})

def snapshot_tensors49(snapshot: HeteroSnapshot) -> dict[str, torch.Tensor]:
    if not isinstance(snapshot, HeteroSnapshot):
        raise ValueError("输入必须是 HeteroSnapshot")
    if snapshot.split not in {"train", "val", "test"}:
        raise ValueError("snapshot split 非法")
    if (not snapshot.tenant_ids or len(set(snapshot.tenant_ids)) != len(snapshot.tenant_ids)
            or any(not isinstance(t, int) or isinstance(t, bool) or t < 0 for t in snapshot.tenant_ids)):
        raise ValueError("snapshot tenant_ids 非法")
    if set(snapshot.x_by_type) != {"user", "item"} or set(snapshot.tenant_by_type) != {"user", "item"}:
        raise ValueError("snapshot 节点类型集合非法")
    if snapshot.item_labels.shape != (snapshot.x_by_type["item"].shape[0],) or snapshot.item_labels.dtype != torch.long:
        raise ValueError("snapshot item_labels 合同不匹配")
    observed_tenants = set(torch.cat([snapshot.tenant_by_type["user"], snapshot.tenant_by_type["item"]]).tolist())
    if observed_tenants != set(snapshot.tenant_ids):
        raise ValueError("snapshot tenant 元数据与张量不一致")
    return {"x.user": snapshot.x_by_type["user"], "x.item": snapshot.x_by_type["item"],
            "tenant.user": snapshot.tenant_by_type["user"], "tenant.item": snapshot.tenant_by_type["item"],
            "label": snapshot.item_labels,
            **{f"edge.{relation_key(k)}": v for k, v in snapshot.edges.items()}}

def recompute_snapshot_digest49(snapshot: HeteroSnapshot) -> str:
    tensors = snapshot_tensors49(snapshot)
    return snapshot_digest(tensors, {"split": snapshot.split, "tenant_ids": snapshot.tenant_ids})

def make_snapshot(split: str, tenant_ids: tuple[int, ...]) -> HeteroSnapshot:
    if split not in {"train", "val", "test"} or not tenant_ids or len(set(tenant_ids)) != len(tenant_ids):
        raise ValueError("split/tenant_ids 非法")
    users, items, user_tenants, item_tenants, labels = [], [], [], [], []
    click_src, click_dst, rev_src, rev_dst, follow_src, follow_dst = [], [], [], [], [], []
    for local, tenant in enumerate(tenant_ids):
        label = tenant % 2
        for role in range(2):
            feature = torch.zeros(8)
            feature[label] = 1.0
            feature[2 + role] = 0.25
            users.append(feature); user_tenants.append(tenant)
            click_src.append(2 * local + role); click_dst.append(local)
            rev_src.append(local); rev_dst.append(2 * local + role)
        follow_src.extend([2 * local, 2 * local + 1])
        follow_dst.extend([2 * local + 1, 2 * local])
        item_feature = torch.zeros(8); item_feature[4] = 1.0
        items.append(item_feature); item_tenants.append(tenant); labels.append(label)
    x_by_type = {"user": torch.stack(users), "item": torch.stack(items)}
    tenant_by_type = {
        "user": torch.tensor(user_tenants, dtype=torch.long),
        "item": torch.tensor(item_tenants, dtype=torch.long),
    }
    edges = {
        click: torch.tensor([click_src, click_dst], dtype=torch.long),
        rev: torch.tensor([rev_src, rev_dst], dtype=torch.long),
        follow: torch.tensor([follow_src, follow_dst], dtype=torch.long),
    }
    item_labels = torch.tensor(labels, dtype=torch.long)
    tensors = {"x.user": x_by_type["user"], "x.item": x_by_type["item"],
               "tenant.user": tenant_by_type["user"], "tenant.item": tenant_by_type["item"],
               "label": item_labels, **{f"edge.{relation_key(k)}": v for k, v in edges.items()}}
    digest = snapshot_digest(tensors, {"split": split, "tenant_ids": tenant_ids})
    return HeteroSnapshot(split, tenant_ids, x_by_type, tenant_by_type, edges, item_labels, digest)

train49 = make_snapshot("train", tuple(range(20)))
val49 = make_snapshot("val", tuple(range(20, 24)))
test49 = make_snapshot("test", tuple(range(24, 28)))
assert train49.x_by_type["user"].shape == (40, 8)
assert train49.x_by_type["item"].shape == (20, 8)
assert train49.edges[click].shape == (2, 40)
assert set(train49.tenant_ids).isdisjoint(val49.tenant_ids)
assert set(val49.tenant_ids).isdisjoint(test49.tenant_ids)
assert len({train49.snapshot_digest, val49.snapshot_digest, test49.snapshot_digest}) == 3
assert recompute_snapshot_digest49(test49) == test49.snapshot_digest


## 6. 分类头、训练协议与张量形状

`HGTNodeClassifier` 先产生 `item:[N_item,D]`，再映射为 `logits:[N_item,2]`。训练只读取 train snapshot；validation 仅用于保存最佳 `state_dict`；test 在冻结后只评一次。由于 batch 很小，复杂度近似 $O(EHD_h + ND^2)$，内存为 $O(ND+EH)$。


In [ ]:
class HGTNodeClassifier(nn.Module):
    def __init__(self, schema: tuple[SchemaEdge, ...], d_model: int = 8, num_heads: int = 2, num_classes: int = 2):
        super().__init__()
        self.layer = HGTLayer(schema, d_model, num_heads)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, snapshot: HeteroSnapshot) -> torch.Tensor:
        hidden = self.layer(snapshot.x_by_type, snapshot.edges, snapshot.tenant_by_type)
        return self.head(hidden["item"])

def accuracy49(model, snapshot):
    model.eval()
    with torch.no_grad():
        pred = model(snapshot).argmax(-1)
    return float((pred == snapshot.item_labels).float().mean())

torch.manual_seed(4903)
model49 = HGTNodeClassifier(SCHEMA)
optimizer49 = torch.optim.Adam(model49.parameters(), lr=0.035)
best_val49, best_state49, best_step49, loss_trace49 = -1.0, None, None, []
for step in range(81):
    model49.train(); optimizer49.zero_grad()
    logits = model49(train49)
    loss = F.cross_entropy(logits, train49.item_labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model49.parameters(), 5.0)
    optimizer49.step()
    loss_trace49.append(float(loss.detach()))
    if step % 5 == 0:
        val_score = accuracy49(model49, val49)
        if val_score > best_val49:
            best_val49 = val_score
            best_state49 = copy.deepcopy(model49.state_dict())
            best_step49 = step
model49.load_state_dict(best_state49)

assert logits.shape == (20, 2)
assert all(torch.isfinite(p).all() for p in model49.parameters())
assert loss_trace49[-1] < loss_trace49[0] * 0.2
assert best_val49 == 1.0
assert best_step49 is not None and best_step49 % 5 == 0
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model49.parameters())


## 7. 冻结评估与反事实检查

除准确率外，再删除 `user→item` 入边做反事实：若预测仍完全不变，通常意味着标签从 item 特征或索引泄漏。这里删除 click 后，item 只能走常量 residual，分类能力应下降。该检查比单看训练 loss 更能说明模型确实使用了目标关系。


In [ ]:
train_acc49 = accuracy49(model49, train49)
val_acc49 = accuracy49(model49, val49)
test_acc49 = accuracy49(model49, test49)
ablated49 = HeteroSnapshot(
    test49.split, test49.tenant_ids, test49.x_by_type, test49.tenant_by_type,
    {rev: test49.edges[rev], follow: test49.edges[follow]}, test49.item_labels, test49.snapshot_digest,
)
ablated_acc49 = accuracy49(model49, ablated49)

assert train_acc49 == 1.0 and val_acc49 == 1.0 and test_acc49 == 1.0
assert ablated_acc49 <= 0.5
assert model49(test49).shape[0] == len(test49.tenant_ids)
assert set(model49.state_dict()) == set(best_state49)


## 8. 发布制品：内部哈希不是信任根

模型包绑定 schema、三份数据 snapshot、tenant split、训练 recipe 与 canonical state digest。state digest 逐个写入 `key/dtype/shape/raw bytes`。loader 返回的不是裸模型，而是 `PublishedHGT` evaluator：每次推理都重算 feature、tenant、edge、label 与 split/tenant metadata 的摘要，再同时核对对象自带摘要和 manifest 登记摘要。

Notebook 中的 `MappingProxyType` 模拟部署时由发布系统/KMS 提供的不可写信任锚；它不是把 registry 塞回 package。若整体替换模型并重算所有内部 digest，仍与发布方登记值不符而 fail-closed。这个 evaluator **只允许登记的 train/val/test 快照**；在线新图推理必须发布独立版本的 schema/feature 服务合同，不能伪造 `snapshot_digest` 冒充离线评测快照。


In [ ]:
RELEASE49 = "hgt-demo-49/v1"
CONFIG49 = {"schema": [list(e) for e in SCHEMA], "d_model": 8, "num_heads": 2, "num_classes": 2}
MANIFEST49 = {
    "config": CONFIG49,
    "snapshots": {"train": train49.snapshot_digest, "val": val49.snapshot_digest, "test": test49.snapshot_digest},
    "split": {"train": list(train49.tenant_ids), "val": list(val49.tenant_ids), "test": list(test49.tenant_ids)},
    "runtime": {"policy": "registered_snapshot_evaluator_only",
                "digest_fields": ["features", "tenants", "edges", "labels", "split", "tenant_ids"]},
    "recipe": {"seed": SEED, "optimizer": "Adam", "lr": 0.035, "steps": 81,
               "grad_clip_norm": 5.0, "validation_interval": 5,
               "selection": "best_validation_accuracy", "tie_break": "first_strict_improvement",
               "checkpoint_step": best_step49},
}
state49 = {k: v.detach().cpu().clone() for k, v in model49.state_dict().items()}
package49 = {"release_id": RELEASE49, "manifest": copy.deepcopy(MANIFEST49), "state": state49}
package49["state_digest"] = tensor_state_digest(package49["state"])
package49["package_digest"] = canonical_digest({
    "release_id": package49["release_id"], "manifest": package49["manifest"], "state_digest": package49["state_digest"]
})
_PUBLISHER_REGISTRY49 = MappingProxyType({RELEASE49: package49["package_digest"]})

class PublishedHGT(nn.Module):
    def __init__(self, model: HGTNodeClassifier, manifest: dict):
        super().__init__()
        self.model = model
        self._manifest = copy.deepcopy(manifest)

    def forward(self, snapshot: HeteroSnapshot) -> torch.Tensor:
        if not isinstance(snapshot, HeteroSnapshot) or snapshot.split not in self._manifest["snapshots"]:
            raise ValueError("仅接受 manifest 登记的 HeteroSnapshot split")
        expected_tenants = tuple(self._manifest["split"][snapshot.split])
        if tuple(snapshot.tenant_ids) != expected_tenants:
            raise ValueError("snapshot tenant split 与发布登记不一致")
        actual_digest = recompute_snapshot_digest49(snapshot)
        if not isinstance(snapshot.snapshot_digest, str) or snapshot.snapshot_digest != actual_digest:
            raise ValueError("snapshot 自带摘要与输入张量/元数据不一致")
        if actual_digest != self._manifest["snapshots"][snapshot.split]:
            raise ValueError("snapshot 未在发布 manifest 登记")
        logits = self.model(snapshot)
        if not torch.isfinite(logits).all():
            raise ValueError("发布模型输出含非有限值")
        return logits

def load_published_hgt(package: dict) -> PublishedHGT:
    required = {"release_id", "manifest", "state", "state_digest", "package_digest"}
    if set(package) != required:
        raise ValueError("package 字段集合非法")
    release_id = package["release_id"]
    if release_id not in _PUBLISHER_REGISTRY49:
        raise ValueError("未知 release，发布方未登记")
    state_hash = tensor_state_digest(package["state"])
    overall = canonical_digest({"release_id": release_id, "manifest": package["manifest"], "state_digest": state_hash})
    if state_hash != package["state_digest"] or overall != package["package_digest"]:
        raise ValueError("包内 canonical digest 校验失败")
    if overall != _PUBLISHER_REGISTRY49[release_id]:
        raise ValueError("publisher registry 信任锚不匹配")
    if package["manifest"] != MANIFEST49:
        raise ValueError("schema/snapshot/split/recipe 合同不匹配")
    cfg = package["manifest"]["config"]
    model = HGTNodeClassifier(tuple(tuple(e) for e in cfg["schema"]), cfg["d_model"], cfg["num_heads"], cfg["num_classes"])
    model.load_state_dict(package["state"], strict=True); model.eval()
    restored = PublishedHGT(model, package["manifest"]); restored.eval()
    return restored

restored49 = load_published_hgt(package49)
assert torch.allclose(restored49(test49), model49(test49), atol=1e-7)
assert isinstance(restored49, PublishedHGT)
assert isinstance(_PUBLISHER_REGISTRY49, MappingProxyType)

fake_digest49 = HeteroSnapshot(test49.split, test49.tenant_ids, test49.x_by_type, test49.tenant_by_type,
                               test49.edges, test49.item_labels, "attacker-digest")
fake_labels_draft49 = HeteroSnapshot(test49.split, test49.tenant_ids, test49.x_by_type, test49.tenant_by_type,
                                    test49.edges, 1 - test49.item_labels, "pending")
fake_labels49 = HeteroSnapshot(fake_labels_draft49.split, fake_labels_draft49.tenant_ids,
                               fake_labels_draft49.x_by_type, fake_labels_draft49.tenant_by_type,
                               fake_labels_draft49.edges, fake_labels_draft49.item_labels,
                               recompute_snapshot_digest49(fake_labels_draft49))
fake_tenants49 = HeteroSnapshot(test49.split, (999,), test49.x_by_type, test49.tenant_by_type,
                                test49.edges, test49.item_labels, test49.snapshot_digest)
snapshot_rejections49 = {}
for attack_name, attacked in {"digest": fake_digest49, "labels": fake_labels49, "tenants": fake_tenants49}.items():
    try:
        restored49(attacked)
        raise AssertionError(f"伪造 snapshot {attack_name} 被接受")
    except ValueError as exc:
        snapshot_rejections49[attack_name] = str(exc)
assert set(snapshot_rejections49) == {"digest", "labels", "tenants"}

forged49 = copy.deepcopy(package49)
forged49["state"]["head.bias"] = forged49["state"]["head.bias"] + 1.0
forged49["state_digest"] = tensor_state_digest(forged49["state"])
forged49["package_digest"] = canonical_digest({"release_id": RELEASE49, "manifest": forged49["manifest"], "state_digest": forged49["state_digest"]})
try:
    load_published_hgt(forged49)
    raise AssertionError("整体替换并重算内部哈希后仍被接受")
except ValueError as exc:
    assert "registry" in str(exc)

try:
    _PUBLISHER_REGISTRY49["evil"] = "digest"
    raise AssertionError("只读 registry 被修改")
except TypeError:
    assert RELEASE49 in _PUBLISHER_REGISTRY49


## 9. 复杂度、失败模式与生产差距

- **复杂度**：关系线性层在朴素实现中约 $O(ED^2)$，注意力归一化约 $O(EH)$；生产会按 relation 分桶、融合 GEMM，并使用 scatter kernel。
- **高频错误**：把 softmax 按 relation 做、反向边复用正向参数、漏检跨 tenant 边、给空邻居输出 bias、把节点随机切分导致同 tenant 泄漏。
- **规模差距**：真实 HGT 需要异构采样、分布式 feature store、热点节点限流、schema 演进与增量索引；本例全图计算只适合 oracle。
- **发布差距**：真实 registry 应在制品之外，由签名/KMS/透明日志保护；还需框架版本、算子 ABI、校准阈值、监控和回滚策略。

至此，模型输出、训练指标、消融与制品恢复都被可执行合同覆盖；任何受控数据上的高分都只作实现证据。


In [ ]:
assert len(SCHEMA) == 3 and len(NODE_TYPES) == 2
assert test_acc49 > ablated_acc49
assert package49["manifest"]["snapshots"]["test"] == test49.snapshot_digest
assert package49["package_digest"] == _PUBLISHER_REGISTRY49[RELEASE49]
assert all(parameter.device.type == "cpu" for parameter in restored49.parameters())
